# CAPA Dashboard - Synthetic Data Generation

Builds a synthetic CAPA dataset on top of cleaned FDA infusion pump recall records.

**Input:** `data/recall_df_clean.csv` — 529 cleaned recall records

**Output:** `data/capa_medfluss.csv` — 529 synthetic CAPA records for MedFluss GmbH

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import random
from datetime import timedelta

In [2]:
DATA_DIR = Path('data')

In [3]:
# Load Data
recall_df = pd.read_csv(DATA_DIR / 'recall_df_clean.csv')
recall_df.shape

(529, 11)

## Define CAPA Dimensions

In [4]:
departments = ['Quality Assurance', 'Regulatory Affairs', 'Clinical Affairs',
                 'Manufacturing', 'Design & Development', 'Supplier Quality']

In [5]:
owners = {
      'Quality Assurance': ['Sarah Müller', 'Thomas Becker'],
      'Regulatory Affairs': ['Priya Nair', 'Klaus Weber'],
      'Clinical Affairs': ['Anna Schmidt', 'David Hoffmann'],
      'Manufacturing': ['Carlos Rivera', 'Lisa Hartmann'],
      'Design & Development': ['Mohammed Al-Farsi', 'Emma Fischer'],
      'Supplier Quality': ['Raj Patel', 'Nina Scholz']
  }

In [6]:
corrected_root_cause_dept_map = {
    'Device Design': 'Design & Development',
    'Software Design': 'Design & Development',
    'Process control': 'Manufacturing',
    'Assembly Error': 'Manufacturing',
    'Human Error — Production': 'Manufacturing',
    'Labeling Execution Error': 'Manufacturing',
    'Nonconforming Material': 'Supplier Quality',
    'Release Without Testing': 'Quality Assurance',
    'Under Investigation': 'Quality Assurance',
    'Labeling Design Error': 'Regulatory Affairs',
    'Documentation Error': 'Regulatory Affairs',
    'Translation Error': 'Regulatory Affairs',
}

recall_df['department'] = recall_df['corrected_root_cause'].map(corrected_root_cause_dept_map)

print(recall_df['department'].value_counts())
print(recall_df['department'].isnull().sum())

department
Design & Development    294
Manufacturing           125
Supplier Quality         93
Quality Assurance         9
Regulatory Affairs        8
Name: count, dtype: int64
0


In [7]:
severity_weights = {
    'Device Design':            {'Critical': 0.08, 'Major': 0.67, 'Minor': 0.25},
    'Software Design':          {'Critical': 0.08, 'Major': 0.67, 'Minor': 0.25},
    'Process control':          {'Critical': 0.06, 'Major': 0.64, 'Minor': 0.30},
    'Assembly Error':           {'Critical': 0.05, 'Major': 0.60, 'Minor': 0.35},
    'Human Error — Production': {'Critical': 0.04, 'Major': 0.56, 'Minor': 0.40},
    'Labeling Execution Error': {'Critical': 0.03, 'Major': 0.52, 'Minor': 0.45},
    'Nonconforming Material':   {'Critical': 0.08, 'Major': 0.64, 'Minor': 0.28},
    'Release Without Testing':  {'Critical': 0.06, 'Major': 0.59, 'Minor': 0.35},
    'Under Investigation':      {'Critical': 0.04, 'Major': 0.51, 'Minor': 0.45},
    'Labeling Design Error':    {'Critical': 0.03, 'Major': 0.57, 'Minor': 0.40},
    'Documentation Error':      {'Critical': 0.02, 'Major': 0.48, 'Minor': 0.50},
    'Translation Error':        {'Critical': 0.02, 'Major': 0.48, 'Minor': 0.50},
}

In [8]:
def assign_severity(root_cause):
      weights = severity_weights.get(root_cause, {'Critical': 0.2, 
                                                  'Major': 0.4, 
                                                  'Minor': 0.4})
      return random.choices(
           list(weights.keys()),
            weights=list(weights.values())
      )[0]

In [9]:
TODAY = pd.Timestamp('2026-05-11')
OVERDUE_THRESHOLD = {'Critical': 30, 'Major': 90, 'Minor': 180}

def assign_open_date():
    start = pd.Timestamp('2022-01-01')
    end   = pd.Timestamp('2026-04-15')
    total_days = (end - start).days
    return start + timedelta(days=random.randint(0, total_days))
    
def assign_capa_status(severity, open_date):
    threshold    = OVERDUE_THRESHOLD[severity]
    days_elapsed = (TODAY - open_date).days

    if days_elapsed < threshold:
        return 'Open' 
    elif days_elapsed < threshold * 1.5:
        return random.choices(['Closed', 'Open'], weights=[0.55, 0.45])[0]
    elif days_elapsed < threshold * 3:
        return random.choices(['Closed', 'Open'], weights=[0.75, 0.25])[0]
    else:
        return random.choices(['Closed', 'Open'], weights=[0.85, 0.15])[0]
        
def assign_close_date(open_date, severity):
    threshold = OVERDUE_THRESHOLD[severity]

    # 10% close fast, 60% close on time, 30% close late
    closure_type = random.choices(['fast', 'normal', 'late'], 
                                  weights=[10, 60, 30])[0]

    if closure_type == 'fast':
        days = random.randint(7, threshold // 2)
    elif closure_type == 'normal':
        days = random.randint(threshold // 2, threshold)
    else:
        days = random.randint(threshold, int(threshold * 1.5))

    close_date = open_date + timedelta(days=days)
    return min(close_date, TODAY - timedelta(days=5))
    
effectiveness_weights = {
    'Critical': [0.50, 0.33, 0.17],
    'Major':    [0.70, 0.20, 0.10],
    'Minor':    [0.82, 0.13, 0.05],
}   

def assign_effectiveness(status, severity):
    if status == 'Open':
        return None
    weights = effectiveness_weights[severity]
    return random.choices(['Effective', 
                           'Partially Effective', 
                           'Not Effective'], 
                           weights=weights)[0]


In [10]:
capa_records = []

for i, row in recall_df.iterrows():
    capa_id     = f'CAPA-MF-{1000 + len(capa_records):04d}'
    root_cause  = row['corrected_root_cause']
    department  = corrected_root_cause_dept_map.get(root_cause, 'Quality Assurance')
    owner       = random.choice(owners[department])
    severity    = assign_severity(root_cause)
    open_date   = assign_open_date()
    capa_status = assign_capa_status(severity, open_date)
    
    if capa_status == 'Closed':
        close_date = assign_close_date(open_date, severity)
        days_open  = (close_date - open_date).days
    else:
        close_date = None
        days_open  = (TODAY - open_date).days
        
    threshold = OVERDUE_THRESHOLD[severity]
    overdue   = 'Yes' if (capa_status == 'Open' and days_open > threshold) else 'No'
    
    capa_records.append({
        'capa_id': capa_id,
        'company': 'MedFluss GmbH',
        'recall_id': row['cfres_id'],
        'product_res_number': row['product_res_number'],
        'recalling_firm': row['recalling_firm'],
        'product_description': row['product_description'],
        'nonconformity': row['reason_for_recall'],
        'original_root_cause': row['root_cause_description'],
        'corrected_root_cause': root_cause,
        'department': department,
        'assigned_owner': owner,
        'severity': severity,
        'capa_status': capa_status,
        'open_date': open_date,
        'close_date': close_date,
        'days_open': days_open,
        'overdue': overdue,
        'corrective_action': row['action'],
        'effectiveness_result': assign_effectiveness(capa_status, severity)
    })

capa_df = pd.DataFrame(capa_records)

In [11]:
# Validate
print(capa_df.shape)
capa_df.head()

(529, 19)


,capa_id,company,recall_id,product_res_number,recalling_firm,product_description,nonconformity,original_root_cause,corrected_root_cause,department,assigned_owner,severity,capa_status,open_date,close_date,days_open,overdue,corrective_action,effectiveness_result
0,CAPA-MF-1000,MedFluss GmbH,85112,Z-0147-2010,Hospira Inc,Power cord for QVue Continuous Cardiac Output ...,Fire/Shock hazard-- The power cord used in the...,Component design/selection,Device Design,Design & Development,Emma Fischer,Major,Closed,2022-06-26,2022-08-22,57,No,Hospira initiated its recall on 08/14/2009. A...,Partially Effective
1,CAPA-MF-1001,MedFluss GmbH,105064,Z-0268-2012,Wolf Medical Supply Inc.,"Wolf Medical Supply, Inc., WOLF-PAK REDI-FLO ...",Redi-Flo Elastomeric Infusion Pumps may have a...,Process control,Process control,Manufacturing,Lisa Hartmann,Minor,Closed,2023-11-24,2024-08-16,266,No,"On 10/20/2011 Wolf Medical Supply Inc., custom...",Effective
2,CAPA-MF-1002,MedFluss GmbH,105401,Z-0269-2012,Wolf Medical Supply Inc.,"Wolf Medical Supply Inc., WOLF-PAK REDI-FLO ...",Redi-Flo Elastomeric Infusion Pumps may have a...,Process control,Process control,Manufacturing,Lisa Hartmann,Major,Closed,2025-09-20,2025-11-17,58,No,"On 10/20/2011 Wolf Medical Supply Inc., custom...",Effective
3,CAPA-MF-1003,MedFluss GmbH,104072,Z-3284-2011,Hospira Inc.,Plum A+ Single Channel Infusion Pumps; Hospira...,Hospira has received reports of incorrect seat...,Nonconforming Material/Component,Nonconforming Material,Supplier Quality,Nina Scholz,Major,Open,2023-05-13,NaT,1094,Yes,"Hospira, Inc. sent an ""URGENT DEVICE RECALL"" l...",NaN
4,CAPA-MF-1004,MedFluss GmbH,107986,Z-1338-2012,Medtronic Neuromodulation,"Medtronic, Model 8870, Application Software Ca...",Medtronic has confirmed that an algorithm used...,Software design,Software Design,Design & Development,Mohammed Al-Farsi,Major,Closed,2025-03-19,2025-05-23,65,No,"Medtronic mailed an ""Urgent Medical Device Cor...",Effective


In [12]:
print("\nSeverity:\n",    capa_df['severity'].value_counts())
print("\nDepartment:\n",  capa_df['department'].value_counts())
print("\nStatus:\n",      capa_df['capa_status'].value_counts())
print("\nOverdue:\n",     capa_df['overdue'].value_counts())
print("\nEffectiveness:\n", capa_df['effectiveness_result'].value_counts(dropna=False))
print("\nDepartment NaN:", capa_df['department'].isna().sum())


Severity:
 severity
Major       344
Minor       147
Critical     38
Name: count, dtype: int64

Department:
 department
Design & Development    294
Manufacturing           125
Supplier Quality         93
Quality Assurance         9
Regulatory Affairs        8
Name: count, dtype: int64

Status:
 capa_status
Closed    424
Open      105
Name: count, dtype: int64

Overdue:
 overdue
No     453
Yes     76
Name: count, dtype: int64

Effectiveness:
 effectiveness_result
Effective              303
NaN                    105
Partially Effective     75
Not Effective           46
Name: count, dtype: int64

Department NaN: 0


In [13]:
print(capa_df.shape)
print(capa_df['severity'].value_counts())
print(capa_df['department'].value_counts())
print(capa_df['overdue'].value_counts())

(529, 19)
severity
Major       344
Minor       147
Critical     38
Name: count, dtype: int64
department
Design & Development    294
Manufacturing           125
Supplier Quality         93
Quality Assurance         9
Regulatory Affairs        8
Name: count, dtype: int64
overdue
No     453
Yes     76
Name: count, dtype: int64


In [14]:
# Save
capa_df.to_csv(DATA_DIR / 'capa_medfluss.csv', index=False)
print("Saved successfully")

Saved successfully
